# Llama-3.2-3B Compound Compression Energy Ablation

Reproducible Google Colab experiment for CSE407 Green Computing.

**Primary model:** `meta-llama/Llama-3.2-3B-Instruct` and `casperhansen/llama-3.2-3b-instruct-awq`  
**Hardware:** one NVIDIA T4 16 GB  
**Primary metrics:** GPU energy, power, latency, throughput, VRAM, NIAH retrieval

## Important implementation status

- C0 and C1 use Hugging Face FP16 dynamic KV caches.
- C2 first attempts the SHA-pinned community TurboQuant cache. It is accepted only
  after shape, packing, sequence-length, and physical-storage checks. If it fails,
  the actual method is labeled `HF_HQQ_4BIT`, a KIVI-inspired fallback.
- C3 applies a SnapKV-style/PyramidKV monkey-patch to `LlamaAttention.forward`.  
  PyramidKV uses a layerwise budget (high in early layers, low in deep layers).
- C4 composes eviction + TurboQuant.

This status is intentional research provenance, not a notebook error.

## 1. Install pinned dependencies

Run this cell once, then use **Runtime > Restart session**, and continue at Section 2.

In [ ]:
%pip install -q "autoawq==0.2.9" "hqq==0.2.8.post1" "nvidia-ml-py==12.575.51" \
    "pandas==2.3.3" "scipy==1.16.3" "statsmodels==0.14.5" \
    "matplotlib==3.10.7" "seaborn==0.13.2" "accelerate==1.11.0"
%pip install -q --upgrade "transformers>=4.46.0,<4.52.0"
%pip install -q --no-deps \
    "git+https://github.com/back2matching/turboquant.git@4de496d48bff9a7524016f60396aa374a999c887"
print("Restart the Colab session before continuing.")

## 2. Imports, Drive, configuration, and environment manifest

In [ ]:
from __future__ import annotations

import contextlib
import csv
import gc
import hashlib
import importlib.metadata
import json
import math
import os
import platform
import random
import re
import threading
import time
import traceback
import uuid
from dataclasses import asdict, dataclass
from datetime import datetime, timezone
from pathlib import Path
from typing import Any, Callable

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scipy.stats as st
import seaborn as sns
import torch
import torch.nn.functional as F
import transformers
from scipy.stats import binomtest, wilcoxon
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    AwqConfig,
    DynamicCache,
)
from transformers.models.llama.modeling_llama import (
    LlamaAttention,
    apply_rotary_pos_emb,
    repeat_kv,
)

try:
    from google.colab import drive
    drive.mount("/content/drive")
    DEFAULT_ROOT = Path("/content/drive/MyDrive/CSE407_Llama3_Energy")
except ImportError:
    DEFAULT_ROOT = Path.cwd() / "llama3_energy_results"

ROOT = Path(os.environ.get("CSE407_RESULTS_DIR", DEFAULT_ROOT))
RAW_DIR = ROOT / "raw"
TRACE_DIR = ROOT / "power_traces"
FIG_DIR = ROOT / "figures"
for directory in (ROOT, RAW_DIR, TRACE_DIR, FIG_DIR):
    directory.mkdir(parents=True, exist_ok=True)

SEED = 407
CONTEXT_LENGTHS = (512, 2048, 8192)
PERFORMANCE_BLOCKS = 10
NIAH_ITEMS_PER_CONDITION = 30
DECODE_TOKENS = 128
NIAH_MAX_NEW_TOKENS = 24
POLL_INTERVAL_S = 0.1
COOLDOWN_TARGET_C = 72
COOLDOWN_TIMEOUT_S = 120
MODEL_BF16 = "unsloth/Llama-3.2-3B-Instruct"
MODEL_AWQ = "casperhansen/llama-3.2-3b-instruct-awq"
TURBOQUANT_COMMIT = "4de496d48bff9a7524016f60396aa374a999c887"

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

RESULT_COLUMNS = [
    "run_id", "timestamp_utc", "experiment", "block", "item_id", "config",
    "requested_method", "actual_method", "method_validated", "model_id",
    "context_target", "input_tokens", "output_tokens",
    "needle_depth", "correct", "response", "prefill_s", "decode_s", "total_s",
    "prefill_tok_s", "decode_tok_s", "avg_power_w", "min_power_w", "max_power_w",
    "gross_energy_j", "idle_power_w", "active_energy_j", "energy_per_output_token_j",
    "torch_loaded_mb", "torch_peak_mb", "nvml_peak_used_mb", "cache_live_mb",
    "prompt_ppl", "power_samples", "trace_file", "valid", "error",
]

def package_version(name: str) -> str | None:
    try:
        return importlib.metadata.version(name)
    except importlib.metadata.PackageNotFoundError:
        return None

def utc_now() -> str:
    return datetime.now(timezone.utc).isoformat()

def gpu_manifest() -> dict[str, Any]:
    if not torch.cuda.is_available():
        raise RuntimeError("A CUDA GPU is required. Select a T4 Colab runtime.")
    props = torch.cuda.get_device_properties(0)
    return {
        "timestamp_utc": utc_now(),
        "python": platform.python_version(),
        "platform": platform.platform(),
        "torch": torch.__version__,
        "transformers": transformers.__version__,
        "autoawq": package_version("autoawq"),
        "turboquant": package_version("turboquant"),
        "hqq": package_version("hqq"),
        "cuda_runtime": torch.version.cuda,
        "gpu_name": props.name,
        "gpu_total_mb": props.total_memory / 2**20,
        "compute_capability": f"{props.major}.{props.minor}",
        "seed": SEED,
        "turboquant_commit": TURBOQUANT_COMMIT,
        "contexts": list(CONTEXT_LENGTHS),
        "performance_blocks": PERFORMANCE_BLOCKS,
        "niah_items_per_condition": NIAH_ITEMS_PER_CONDITION,
        "baseline_dtype": "float16",
        "model_bf16": MODEL_BF16,
        "model_awq": MODEL_AWQ,
    }

MANIFEST = gpu_manifest()
if "T4" not in MANIFEST["gpu_name"]:
    raise RuntimeError(f"Expected an NVIDIA T4, found {MANIFEST['gpu_name']!r}.")
print("\n".join(f"{k}: {v}" for k, v in MANIFEST.items()))

## 3. NVML power monitor

Energy is trapezoidally integrated from timestamped samples at 10 Hz.

In [ ]:
import pynvml

pynvml.nvmlInit()
NVML_HANDLE = pynvml.nvmlDeviceGetHandleByIndex(0)

def append_csv_atomic(path: Path, row: dict[str, Any], columns=RESULT_COLUMNS) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    lock = path.with_suffix(path.suffix + ".lock")
    while True:
        try:
            fd = os.open(lock, os.O_CREAT | os.O_EXCL | os.O_WRONLY)
            os.close(fd)
            break
        except FileExistsError:
            time.sleep(0.05)
    try:
        exists = path.exists() and path.stat().st_size > 0
        with path.open("a", newline="", encoding="utf-8") as handle:
            writer = csv.DictWriter(handle, fieldnames=columns, extrasaction="ignore")
            if not exists:
                writer.writeheader()
            writer.writerow({key: row.get(key) for key in columns})
            handle.flush()
            os.fsync(handle.fileno())
    finally:
        lock.unlink(missing_ok=True)

class GPUMonitor:
    def __init__(self, interval_s: float = POLL_INTERVAL_S):
        self.interval_s = interval_s
        self.samples: list[dict[str, float]] = []
        self._running = False
        self._thread: threading.Thread | None = None
        self._t0 = 0.0

    def _sample(self) -> None:
        try:
            memory = pynvml.nvmlDeviceGetMemoryInfo(NVML_HANDLE)
            self.samples.append({
                "t_s": time.perf_counter() - self._t0,
                "power_w": pynvml.nvmlDeviceGetPowerUsage(NVML_HANDLE) / 1000.0,
                "temperature_c": float(pynvml.nvmlDeviceGetTemperature(NVML_HANDLE, pynvml.NVML_TEMPERATURE_GPU)),
                "gpu_util_pct": float(pynvml.nvmlDeviceGetUtilizationRates(NVML_HANDLE).gpu),
                "memory_used_mb": memory.used / 2**20,
            })
        except pynvml.NVMLError as exc:
            self.samples.append({"t_s": time.perf_counter() - self._t0, "error": str(exc)})

    def _poll(self) -> None:
        deadline = self._t0
        while self._running:
            self._sample()
            deadline += self.interval_s
            time.sleep(max(0.0, deadline - time.perf_counter()))

    def start(self) -> None:
        torch.cuda.synchronize()
        torch.cuda.reset_peak_memory_stats()
        self.samples = []
        self._running = True
        self._t0 = time.perf_counter()
        self._thread = threading.Thread(target=self._poll, daemon=True)
        self._thread.start()

    def stop(self, idle_power_w: float) -> dict[str, float]:
        torch.cuda.synchronize()
        elapsed = time.perf_counter() - self._t0
        self._running = False
        if self._thread:
            self._thread.join(timeout=2)
        self._sample()
        valid = [sample for sample in self.samples if "power_w" in sample]
        if len(valid) < 2:
            raise RuntimeError(f"Only {len(valid)} valid NVML power samples were collected.")
        times = np.asarray([sample["t_s"] for sample in valid], dtype=float)
        powers = np.asarray([sample["power_w"] for sample in valid], dtype=float)
        gross_j = float(getattr(np, "trapezoid", np.trapz)(powers, times))
        return {
            "monitor_elapsed_s": elapsed,
            "avg_power_w": float(gross_j / max(times[-1] - times[0], 1e-9)),
            "min_power_w": float(powers.min()),
            "max_power_w": float(powers.max()),
            "gross_energy_j": gross_j,
            "active_energy_j": max(0.0, gross_j - idle_power_w * (times[-1] - times[0])),
            "nvml_peak_used_mb": max(sample["memory_used_mb"] for sample in valid),
            "power_samples": len(valid),
        }

def measure_idle_power(seconds: float = 30.0) -> float:
    samples = []
    end = time.monotonic() + seconds
    while time.monotonic() < end:
        samples.append(pynvml.nvmlDeviceGetPowerUsage(NVML_HANDLE) / 1000.0)
        time.sleep(POLL_INTERVAL_S)
    idle = float(np.median(samples))
    (ROOT / "idle_power.json").write_text(
        json.dumps({"timestamp_utc": utc_now(), "seconds": seconds, "median_w": idle, "samples_w": samples}, indent=2),
        encoding="utf-8",
    )
    return idle

IDLE_POWER_W = measure_idle_power(30.0)
print(f"Idle median: {IDLE_POWER_W:.2f} W")

## 4. Exact-token NIAH inputs

Inputs use a sentinel in Llama's chat template for exact token count control.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_BF16, trust_remote_code=True)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token_id = tokenizer.eos_token_id

SENTINEL = "<CSE407_INSERT_9f31>"
FILLER_TEXT = (
    "The history of computing is a fascinating journey through time. "
    "From the earliest mechanical calculators to modern quantum computers, "
    "humanity has constantly pushed the boundaries of what machines can do. "
)
NEEDLE = "The secret project code name is SHADOW-8821."

def template_parts(instruction: str) -> tuple[list[int], list[int]]:
    messages = [
        {"role": "system", "content": "Extract the exact information requested."},
        {"role": "user", "content": f"{instruction}\n\n{SENTINEL}"},
    ]
    rendered = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )
    if rendered.count(SENTINEL) != 1:
        raise RuntimeError("Sentinel not preserved exactly once in chat template.")
    left, right = rendered.split(SENTINEL)
    return tokenizer.encode(left, add_special_tokens=False), tokenizer.encode(right, add_special_tokens=False)

def repeated_ids(text: str, count: int) -> list[int]:
    unit = tokenizer.encode(text, add_special_tokens=False)
    if not unit:
        raise ValueError("Filler text tokenized to empty sequence.")
    return (unit * math.ceil(count / len(unit)))[:count]

def make_niah_input(context_length: int, depth: float, item_id: int) -> dict[str, torch.Tensor]:
    instruction = (
        "Read the document below. Complete the final statement using "
        "only the code name from the document. Return the code name and nothing else."
    )
    left, right = template_parts(instruction)
    needle_ids = tokenizer.encode(f"\n{NEEDLE}\n", add_special_tokens=False)
    question_ids = tokenizer.encode("\nThe secret project code name is", add_special_tokens=False)
    available = context_length - len(left) - len(right) - len(needle_ids) - len(question_ids)
    if available < 32:
        raise ValueError(f"Context {context_length} too short for fixed prompt overhead.")
    rng = random.Random(SEED * 100000 + context_length * 100 + item_id)
    jitter = rng.randint(-max(1, available // 100), max(1, available // 100))
    before = min(available, max(0, round(available * depth) + jitter))
    after = available - before
    ids = left + repeated_ids(FILLER_TEXT, before) + needle_ids + repeated_ids(FILLER_TEXT, after) + question_ids + right
    if len(ids) != context_length:
        raise AssertionError(f"Constructed {len(ids)} tokens, expected {context_length}.")
    return {
        "input_ids": torch.tensor([ids], dtype=torch.long),
        "attention_mask": torch.ones((1, context_length), dtype=torch.long),
    }

def niah_correct(response: str) -> bool:
    return "shadow-8821" in response.lower()

for length in CONTEXT_LENGTHS:
    assert make_niah_input(length, 0.5, 0)["input_ids"].shape[1] == length
print("Exact-token prompt checks passed.")

## 5. Cache factories, eviction patch, capability registry

Llama 3.2-3B: 28 layers, 24 Q heads, 8 KV heads, head_dim=128.

In [ ]:
def tensor_storage_bytes(root: Any) -> int:
    seen_objects: set[int] = set()
    seen_storages: set[tuple[str, int]] = set()
    total = 0
    def visit(value: Any) -> None:
        nonlocal total
        object_id = id(value)
        if object_id in seen_objects:
            return
        seen_objects.add(object_id)
        if isinstance(value, torch.Tensor):
            if value.numel() == 0:
                return
            storage = value.untyped_storage()
            key = (str(value.device), storage.data_ptr())
            if key not in seen_storages:
                seen_storages.add(key)
                total += storage.nbytes()
            return
        if isinstance(value, dict):
            for child in value.values():
                visit(child)
        elif isinstance(value, (list, tuple, set)):
            for child in value:
                visit(child)
        elif hasattr(value, "__dict__"):
            for child in vars(value).values():
                visit(child)
    visit(root)
    return total

# ---- PyramidKV budget ----
# Llama 3.2-3B: 28 layers
# Budget: early layers get more, late layers get less
# Total = 28 * 1024 = 28672 layer-token entries
NUM_LAYERS = 28
NUM_Q_HEADS = 24
NUM_KV_HEADS = 8
HEAD_DIM = 128

def compute_pyramid_budgets(num_layers: int, avg_budget: int = 1024) -> list[int]:
    total = num_layers * avg_budget
    ramp = np.linspace(1.5, 0.5, num_layers)
    weights = ramp / ramp.sum()
    budgets = np.round(weights * total).astype(int)
    diff = total - budgets.sum()
    budgets[-1] += diff
    return budgets.tolist()

PYRAMID_BUDGETS = compute_pyramid_budgets(NUM_LAYERS)
print(f"PyramidKV budgets (sum={sum(PYRAMID_BUDGETS)}, avg={sum(PYRAMID_BUDGETS)//NUM_LAYERS}):")
for i in range(0, NUM_LAYERS, 7):
    print(f"  L{i:02d}-L{min(i+6, NUM_LAYERS-1):02d}: {PYRAMID_BUDGETS[i:i+7]}")

# ---- Eviction state ----
_EVICTION_ENABLED = False

def set_eviction(enabled: bool):
    global _EVICTION_ENABLED
    _EVICTION_ENABLED = enabled

def make_eviction_forward():
    OBS_WINDOW = 32
    SINK_TOKENS = 4

    def eviction_forward(
        self,
        hidden_states: torch.Tensor,
        attention_mask: torch.Tensor | None = None,
        position_ids: torch.LongTensor | None = None,
        past_key_value: DynamicCache | None = None,
        output_attentions: bool = False,
        use_cache: bool = False,
        cache_position: torch.LongTensor | None = None,
        position_embeddings: tuple | None = None,
        **kwargs,
    ):
        bsz, q_len, _ = hidden_states.size()
        num_heads = self.num_heads
        num_kv_heads = self.num_key_value_heads
        head_dim = self.head_dim
        num_kv_groups = num_heads // num_kv_heads

        query_states = self.q_proj(hidden_states)
        key_states = self.k_proj(hidden_states)
        value_states = self.v_proj(hidden_states)

        query_states = query_states.view(bsz, q_len, num_heads, head_dim).transpose(1, 2)
        key_states = key_states.view(bsz, q_len, num_kv_heads, head_dim).transpose(1, 2)
        value_states = value_states.view(bsz, q_len, num_kv_heads, head_dim).transpose(1, 2)

        if position_embeddings is not None:
            cos, sin = position_embeddings
        else:
            cos, sin = self.rotary_emb(value_states, position_ids)
        query_states, key_states = apply_rotary_pos_emb(query_states, key_states, cos, sin)

        layer_budget = PYRAMID_BUDGETS[self.layer_idx]
        total_len = (past_key_value.get_seq_length() or 0) + q_len
        should_evict = (
            _EVICTION_ENABLED
            and past_key_value is not None
            and total_len > layer_budget
        )

        if should_evict:
            past_keys, past_values = past_key_value.update(
                key_states, value_states, self.layer_idx, {"cache_position": cache_position}
            )
            obs_queries = query_states[:, :, -OBS_WINDOW:, :]
            scores = torch.matmul(
                obs_queries, past_keys.transpose(2, 3)
            ).sum(dim=2).sum(dim=1) / math.sqrt(head_dim)
            scores[:, :SINK_TOKENS] = 1e9
            scores[:, -OBS_WINDOW:] = 1e9
            _, top_indices = torch.topk(scores, layer_budget, dim=-1)
            keep_indices, _ = torch.sort(top_indices, dim=-1)
            past_key_value._key_cache[self.layer_idx] = past_keys[:, :, keep_indices[0], :].contiguous()
            past_key_value._value_cache[self.layer_idx] = past_values[:, :, keep_indices[0], :].contiguous()
            key_states = past_key_value._key_cache[self.layer_idx]
            value_states = past_key_value._value_cache[self.layer_idx]
        else:
            if past_key_value is not None:
                key_states, value_states = past_key_value.update(
                    key_states, value_states, self.layer_idx, {"cache_position": cache_position}
                )

        key_states_rep = repeat_kv(key_states, num_kv_groups)
        value_states_rep = repeat_kv(value_states, num_kv_groups)
        attn_weights = torch.matmul(query_states, key_states_rep.transpose(2, 3)) / math.sqrt(head_dim)
        if attention_mask is not None:
            if attention_mask.shape[-1] > attn_weights.shape[-1]:
                diff = attention_mask.shape[-1] - attn_weights.shape[-1]
                attention_mask = attention_mask[:, :, :, diff:]
            attn_weights = attn_weights + attention_mask
        attn_weights = F.softmax(attn_weights, dim=-1, dtype=torch.float32).to(query_states.dtype)
        attn_output = torch.matmul(attn_weights, value_states_rep)

        attn_output = attn_output.transpose(1, 2).contiguous()
        attn_output = attn_output.reshape(bsz, q_len, -1)
        attn_output = self.o_proj(attn_output)

        if output_attentions:
            return attn_output, attn_weights, past_key_value
        return attn_output, None, past_key_value

    return eviction_forward

_PATCHED_FORWARD = make_eviction_forward()
_ORIG_FORWARD = LlamaAttention.forward

def apply_eviction_patch():
    LlamaAttention.forward = _PATCHED_FORWARD

def remove_eviction_patch():
    LlamaAttention.forward = _ORIG_FORWARD

# ---- Standard cache factories ----
def full_cache(model):
    return DynamicCache()

def hqq_cache(model):
    from transformers.cache_utils import QuantizedCache, QuantizedCacheConfig
    return QuantizedCache(QuantizedCacheConfig(
        backend="hqq",
        nbits=4,
        axis_key=1,
        axis_value=1,
        q_group_size=64,
        residual_length=128,
        compute_dtype=torch.float16,
    ))

def turboquant_cache(model):
    from turboquant import TurboQuantCache
    return TurboQuantCache(bits=4)

# ---- Config registry ----
@dataclass(frozen=True)
class ExperimentConfig:
    code: str
    model_id: str
    requested_method: str
    cache_factory: Callable[[Any], Any] | None
    actual_method: str
    method_validated: bool
    use_eviction: bool = False
    blocked_reason: str = ""

CONFIGS: dict[str, ExperimentConfig] = {}

def register_config(code, model_id, requested, cache_factory, actual, validated, eviction=False, reason=""):
    CONFIGS[code] = ExperimentConfig(code, model_id, requested, cache_factory, actual, validated, eviction, reason)

# Temporary — will be updated after pilots
register_config("C0", MODEL_BF16, "FullKV", full_cache, "FullKV", True)
register_config("C1", MODEL_AWQ, "FullKV", full_cache, "FullKV", True)
register_config("C2", MODEL_AWQ, "TurboQuant-4bit", turboquant_cache, "pending", False)
register_config("C3", MODEL_AWQ, "PyramidKV-1024", full_cache, "pending", False, eviction=True)
register_config("C4", MODEL_AWQ, "PyramidKV-1024+TurboQuant-4bit", turboquant_cache, "pending", False, eviction=True)

def display_capabilities() -> pd.DataFrame:
    return pd.DataFrame([
        {
            "config": cfg.code,
            "model": cfg.model_id.split("/")[-1],
            "requested": cfg.requested_method,
            "actual": cfg.actual_method,
            "validated": cfg.method_validated,
            "eviction": cfg.use_eviction,
            "status": "ready" if cfg.method_validated else (cfg.blocked_reason or "pending"),
        }
        for cfg in CONFIGS.values()
    ])

display(display_capabilities())

## 6. Model loading and cache pilots

Each config is piloted at 512 tokens. Only configs that pass all gates may proceed.

In [ ]:
def clear_cuda() -> None:
    for _ in range(3):
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
            torch.cuda.synchronize()

def load_model(model_id: str):
    clear_cuda()
    kwargs = {
        "device_map": {"": 0},
        "attn_implementation": "eager",
        "trust_remote_code": True,
    }
    if "unsloth" in model_id or "meta-llama" in model_id:
        kwargs["torch_dtype"] = torch.float16
    if "awq" in model_id.lower():
        kwargs["quantization_config"] = AwqConfig(fuse_modules=True)
    model = AutoModelForCausalLM.from_pretrained(model_id, **kwargs).eval()
    if model.config.model_type != "llama":
        raise RuntimeError(f"Expected model_type='llama', got {model.config.model_type!r}.")
    cfg = model.config
    assert cfg.num_hidden_layers == NUM_LAYERS, f"Expected {NUM_LAYERS} layers, got {cfg.num_hidden_layers}"
    assert cfg.num_attention_heads == NUM_Q_HEADS, f"Expected {NUM_Q_HEADS} Q heads, got {cfg.num_attention_heads}"
    assert cfg.num_key_value_heads == NUM_KV_HEADS, f"Expected {NUM_KV_HEADS} KV heads, got {cfg.num_key_value_heads}"
    assert cfg.hidden_size // cfg.num_attention_heads == HEAD_DIM, f"Expected head_dim={HEAD_DIM}"
    return model

def validate_full_cache(model) -> dict[str, Any]:
    inputs = make_niah_input(512, 0.5, 999)
    inputs = {key: value.to(model.device) for key, value in inputs.items()}
    cache = full_cache(model)
    with torch.inference_mode():
        output = model(**inputs, past_key_values=cache, use_cache=True)
    assert output.past_key_values is cache
    assert cache.get_seq_length() is None or cache.get_seq_length() >= 512
    live = tensor_storage_bytes(cache)
    assert live > 0, "Cache storage must be positive"
    return {"actual_method": "FullKV", "cache_live_bytes": live}

def validate_turboquant_or_fallback(model) -> dict[str, Any]:
    inputs = make_niah_input(512, 0.5, 998)
    inputs = {key: value.to(model.device) for key, value in inputs.items()}
    errors = []
    try:
        cache = turboquant_cache(model)
        with torch.inference_mode():
            output = model(**inputs, past_key_values=cache, use_cache=True)
        assert output.past_key_values is cache
        live = tensor_storage_bytes(cache)
        fp16_bytes = (
            NUM_LAYERS * NUM_KV_HEADS * 512 * HEAD_DIM * 2 * 2
        )
        if live >= fp16_bytes:
            raise AssertionError(
                f"TurboQuant: reachable storage ({live/2**20:.1f} MiB) is not below "
                f"FP16 equivalent ({fp16_bytes/2**20:.1f} MiB)."
            )
        return {
            "actual_method": "TurboQuant-4bit",
            "cache_factory": turboquant_cache,
            "cache_live_bytes": live,
            "errors": errors,
        }
    except Exception as exc:
        errors.append(f"TurboQuant rejected: {type(exc).__name__}: {exc}")
        clear_cuda()
    cache = hqq_cache(model)
    with torch.inference_mode():
        output = model(**inputs, past_key_values=cache, use_cache=True)
    live = tensor_storage_bytes(cache)
    fp16_bytes = NUM_LAYERS * NUM_KV_HEADS * 512 * HEAD_DIM * 2 * 2
    if live >= fp16_bytes:
        raise AssertionError(f"HQQ fallback did not reduce storage below FP16 ({live/2**20:.1f} vs {fp16_bytes/2**20:.1f} MiB).")
    return {
        "actual_method": "HF_HQQ_4BIT",
        "cache_factory": hqq_cache,
        "cache_live_bytes": live,
        "errors": errors,
    }

def validate_eviction(model) -> dict[str, Any]:
    apply_eviction_patch()
    set_eviction(True)
    try:
        inputs = make_niah_input(512, 0.5, 997)
        inputs = {key: value.to(model.device) for key, value in inputs.items()}
        cache = full_cache(model)
        with torch.inference_mode():
            output = model(**inputs, past_key_values=cache, use_cache=True)
        live = tensor_storage_bytes(cache)
        assert live > 0
        return {"actual_method": "PyramidKV-1024", "cache_factory": full_cache, "cache_live_bytes": live}
    finally:
        remove_eviction_patch()
        set_eviction(False)

# ---- Run pilots ----
pilot_rows = []

bf16_model = load_model(MODEL_BF16)
pilot_rows.append({"config": "C0", **validate_full_cache(bf16_model)})
del bf16_model; clear_cuda()

awq_model = load_model(MODEL_AWQ)
pilot_rows.append({"config": "C1", **validate_full_cache(awq_model)})

c2_pilot = validate_turboquant_or_fallback(awq_model)
pilot_rows.append({"config": "C2", "actual_method": c2_pilot["actual_method"],
    "cache_live_bytes": c2_pilot["cache_live_bytes"], "errors": "; ".join(c2_pilot.get("errors", []))})

c2_factory = c2_pilot["cache_factory"]
c2_method = c2_pilot["actual_method"]
c2_validated = "TurboQuant" in c2_method
CONFIGS["C2"] = ExperimentConfig("C2", MODEL_AWQ, "TurboQuant-4bit", c2_factory, c2_method, c2_validated)

c3_pilot = validate_eviction(awq_model)
pilot_rows.append({"config": "C3", **c3_pilot})
CONFIGS["C3"] = ExperimentConfig("C3", MODEL_AWQ, "PyramidKV-1024", full_cache, "PyramidKV-1024", True, eviction=True)

if c2_validated:
    CONFIGS["C4"] = ExperimentConfig(
        "C4", MODEL_AWQ, "PyramidKV-1024+TurboQuant-4bit",
        c2_factory, f"PyramidKV-1024+{c2_method}", True, eviction=True,
    )
    pilot_rows.append({"config": "C4", "actual_method": CONFIGS["C4"].actual_method,
        "cache_live_bytes": c2_pilot["cache_live_bytes"]})
else:
    CONFIGS["C4"] = ExperimentConfig(
        "C4", MODEL_AWQ, "PyramidKV-1024+TurboQuant-4bit", None,
        "BLOCKED", False, blocked_reason="C2 (TurboQuant) did not pass validation.",
    )
    pilot_rows.append({"config": "C4", "actual_method": "BLOCKED", "cache_live_bytes": 0})

del awq_model; clear_cuda()

(ROOT / "cache_pilots.json").write_text(json.dumps(pilot_rows, indent=2, default=str), encoding="utf-8")
display(pd.DataFrame(pilot_rows))
display(display_capabilities())

## 7. Timed prefill and decode runner

Performance and NIAH runs with cooldown between conditions.

In [ ]:
def wait_for_cooldown(target_c: int = COOLDOWN_TARGET_C, timeout_s: int = COOLDOWN_TIMEOUT_S) -> float:
    start = time.monotonic()
    while time.monotonic() - start < timeout_s:
        temperature = pynvml.nvmlDeviceGetTemperature(NVML_HANDLE, pynvml.NVML_TEMPERATURE_GPU)
        if temperature <= target_c:
            return float(temperature)
        time.sleep(2)
    return float(pynvml.nvmlDeviceGetTemperature(NVML_HANDLE, pynvml.NVML_TEMPERATURE_GPU))

def decode_response(token_ids: torch.Tensor) -> str:
    return tokenizer.decode(token_ids.tolist(), skip_special_tokens=True, clean_up_tokenization_spaces=False)

def measured_generate(
    model,
    cfg: ExperimentConfig,
    inputs_cpu: dict[str, torch.Tensor],
    max_new_tokens: int,
    trace_stem: str,
) -> dict[str, Any]:
    if cfg.cache_factory is None or not cfg.method_validated:
        raise RuntimeError(f"{cfg.code} is not executable: {cfg.blocked_reason}")
    wait_for_cooldown()
    clear_cuda()
    inputs = {key: value.to(model.device) for key, value in inputs_cpu.items()}

    if cfg.use_eviction:
        apply_eviction_patch()
        set_eviction(True)
    else:
        remove_eviction_patch()
        set_eviction(False)

    cache = cfg.cache_factory(model)
    loaded_mb = torch.cuda.memory_allocated() / 2**20
    monitor = GPUMonitor()
    monitor.start()
    started = time.perf_counter()
    try:
        prefill_started = time.perf_counter()
        with torch.inference_mode():
            output = model(**inputs, past_key_values=cache, use_cache=True)
        torch.cuda.synchronize()
        prefill_s = time.perf_counter() - prefill_started

        logits = output.logits[0]
        input_ids = inputs["input_ids"][0]
        shift_logits = logits[:-1, :].contiguous()
        shift_labels = input_ids[1:].contiguous()
        prompt_loss = F.cross_entropy(shift_logits, shift_labels)
        prompt_ppl = float(torch.exp(prompt_loss))

        next_token = output.logits[:, -1, :].argmax(dim=-1, keepdim=True)
        generated = []
        attention_mask = inputs["attention_mask"]
        decode_started = time.perf_counter()
        for _ in range(max_new_tokens):
            generated.append(next_token)
            if next_token.item() == tokenizer.eos_token_id:
                break
            attention_mask = torch.cat(
                [attention_mask, torch.ones((1, 1), dtype=attention_mask.dtype, device=model.device)],
                dim=1,
            )
            with torch.inference_mode():
                output = model(
                    input_ids=next_token,
                    attention_mask=attention_mask,
                    past_key_values=cache,
                    use_cache=True,
                    
                )
            next_token = output.logits[:, -1, :].argmax(dim=-1, keepdim=True)
        torch.cuda.synchronize()
        decode_s = time.perf_counter() - decode_started
        total_s = time.perf_counter() - started
        power = monitor.stop(IDLE_POWER_W)
    except Exception:
        with contextlib.suppress(Exception):
            monitor.stop(IDLE_POWER_W)
        raise
    finally:
        if cfg.use_eviction:
            remove_eviction_patch()
            set_eviction(False)

    output_ids = torch.cat(generated, dim=1)[0] if generated else torch.empty(0, dtype=torch.long, device=model.device)
    response = decode_response(output_ids.detach().cpu())
    cache_live_mb = tensor_storage_bytes(cache) / 2**20
    trace_path = TRACE_DIR / f"{trace_stem}.jsonl"
    with trace_path.open("w", encoding="utf-8") as handle:
        for sample in monitor.samples:
            handle.write(json.dumps(sample) + "\n")
    output_tokens = int(output_ids.numel())
    return {
        "response": response,
        "input_tokens": int(inputs["input_ids"].shape[1]),
        "output_tokens": output_tokens,
        "prefill_s": prefill_s,
        "decode_s": decode_s,
        "total_s": total_s,
        "prefill_tok_s": int(inputs["input_ids"].shape[1]) / max(prefill_s, 1e-9),
        "decode_tok_s": output_tokens / max(decode_s, 1e-9),
        "torch_loaded_mb": loaded_mb,
        "torch_peak_mb": torch.cuda.max_memory_allocated() / 2**20,
        "cache_live_mb": cache_live_mb,
        "prompt_ppl": prompt_ppl,
        "trace_file": str(trace_path.relative_to(ROOT)),
        "energy_per_output_token_j": power["gross_energy_j"] / max(output_tokens, 1),
        **power,
    }

def base_result(cfg: ExperimentConfig, experiment: str, context: int, **extra) -> dict[str, Any]:
    return {
        "run_id": str(uuid.uuid4()),
        "timestamp_utc": utc_now(),
        "experiment": experiment,
        "config": cfg.code,
        "requested_method": cfg.requested_method,
        "actual_method": cfg.actual_method,
        "method_validated": cfg.method_validated,
        "model_id": cfg.model_id,
        "context_target": context,
        "idle_power_w": IDLE_POWER_W,
        "valid": False,
        "error": "",
        **extra,
    }

def run_and_save(model, cfg: ExperimentConfig, inputs, output_path: Path, trace_stem: str, **metadata):
    row = base_result(cfg, metadata.pop("experiment"), metadata.pop("context_target"), **metadata)
    try:
        row.update(measured_generate(model, cfg, inputs, metadata.get("max_new_tokens", DECODE_TOKENS), trace_stem))
        row["valid"] = (
            row["power_samples"] >= 2
            and 5.0 <= row["min_power_w"] <= row["max_power_w"] <= 100.0
            and row["input_tokens"] == row["context_target"]
        )
    except Exception as exc:
        row["error"] = f"{type(exc).__name__}: {exc}"
        traceback.print_exc()
    append_csv_atomic(output_path, row)
    return row

## 8. Mandatory sanity checks

C0 must retrieve the needle at 512 tokens. Stop if these fail.

In [ ]:
from transformers import set_seed
set_seed(SEED)

output_path = RAW_DIR / "sanity_results.csv"

for sanity_cfg_code in ["C0", "C1"]:
    cfg = CONFIGS[sanity_cfg_code]
    model = load_model(cfg.model_id)
    for ctx in [512]:
        inputs = make_niah_input(ctx, 0.5, 0)
        row = run_and_save(
            model, cfg, inputs, output_path,
            trace_stem=f"sanity_{cfg.code}_{ctx}",
            experiment="sanity", block=0, item_id=0,
            context_target=ctx, thinking=False, needle_depth=0.5,
            max_new_tokens=NIAH_MAX_NEW_TOKENS,
        )
        ok = row.get("valid", False) and niah_correct(row.get("response", ""))
        print(f"{cfg.code} @ {ctx}: valid={row.get('valid')}, needle={niah_correct(row.get('response', ''))}")
        if not ok:
            raise RuntimeError(f"Sanity check failed: {cfg.code} @ {ctx}")
    del model; clear_cuda()

print("All sanity checks passed.")

## 9. Full NIAH experiment matrix

Runs every validated config at every context length.

In [ ]:
RUN_FULL = True
NUM_REPETITIONS = 3
niah_output_path = RAW_DIR / "niah_results.csv"

if not RUN_FULL:
    print("Full run disabled. Set RUN_FULL = True to execute.")
else:
    for cfg_code, cfg in CONFIGS.items():
        if not cfg.method_validated:
            print(f"Skipping {cfg_code}: {cfg.blocked_reason or 'not validated'}")
            continue
        model = load_model(cfg.model_id)
        for ctx in CONTEXT_LENGTHS:
            for rep in range(NUM_REPETITIONS):
                for depth, depth_label in [(0.25, "early"), (0.5, "mid"), (0.75, "late")]:
                    for item_id in range(10):
                        global_id = rep * 10000 + int(depth * 100) * 100 + item_id
                        inputs = make_niah_input(ctx, depth, global_id)
                        row = run_and_save(
                            model, cfg, inputs, niah_output_path,
                            trace_stem=f"niah_{cfg_code}_{ctx}_r{rep}_d{depth_label}_{item_id}",
                            experiment="niah", block=rep, item_id=f"d{depth_label}_{item_id}",
                            context_target=ctx, thinking=False, needle_depth=depth,
                            max_new_tokens=NIAH_MAX_NEW_TOKENS,
                        )
                        ok = row.get("valid", False) and niah_correct(row.get("response", ""))
                        print(f"{cfg_code} @ {ctx} rep={rep} depth={depth_label} item={item_id}: {'OK' if ok else 'FAIL'}")
        del model; clear_cuda()
    print("NIAH experiment complete.")

## 10. Performance experiment matrix

In [ ]:
perf_output_path = RAW_DIR / "perf_results.csv"

if not RUN_FULL:
    print("Full run disabled.")
else:
    for cfg_code, cfg in CONFIGS.items():
        if not cfg.method_validated:
            print(f"Skipping {cfg_code}: {cfg.blocked_reason or 'not validated'}")
            continue
        model = load_model(cfg.model_id)
        for ctx in CONTEXT_LENGTHS:
            for rep in range(NUM_REPETITIONS):
                inputs = make_niah_input(ctx, 0.5, -1)  # fixed position
                row = run_and_save(
                    model, cfg, inputs, perf_output_path,
                    trace_stem=f"perf_{cfg_code}_{ctx}_r{rep}",
                    experiment="performance", block=rep, item_id="",
                    context_target=ctx, thinking=False, needle_depth=0.5,
                    max_new_tokens=DECODE_TOKENS,
                )
                print(f"{cfg_code} @ {ctx} rep={rep}: {row.get('total_s', 0):.1f}s, "
                    f"{row.get('gross_energy_j', 0):.1f}J, "
                    f"VRAM={row.get('torch_peak_mb', 0):.0f}MB")
        del model; clear_cuda()
    print("Performance experiment complete.")

## 11. Results analysis and visualization

In [ ]:
def load_results(path: Path) -> pd.DataFrame:
    if not path.exists():
        print(f"No results at {path}")
        return pd.DataFrame()
    df = pd.read_csv(path)
    df = df[df["valid"] == True].copy()
    return df

df_niah = load_results(niah_output_path)
df_perf = load_results(perf_output_path)

def plot_results(df_perf: pd.DataFrame, df_niah: pd.DataFrame):
    if df_perf.empty and df_niah.empty:
        print("No data to plot. Run the experiment first.")
        return

    sns.set_style("whitegrid")
    fig, axes = plt.subplots(2, 3, figsize=(18, 12))

    if not df_perf.empty:
        ax = axes[0, 0]
        for cfg in df_perf["config"].unique():
            d = df_perf[df_perf["config"] == cfg].groupby("context_target")["torch_peak_mb"].mean()
            ax.plot(d.index, d.values, "o-", label=cfg, lw=2, ms=8)
        ax.set_xlabel("Context Length"); ax.set_ylabel("Peak VRAM (MB)")
        ax.set_title("GPU Memory vs Context Length"); ax.legend(); ax.grid(alpha=0.3)

        ax = axes[0, 1]
        for cfg in df_perf["config"].unique():
            d = df_perf[df_perf["config"] == cfg].groupby("context_target")["gross_energy_j"].mean()
            ax.plot(d.index, d.values, "s-", label=cfg, lw=2, ms=8)
        ax.set_xlabel("Context Length"); ax.set_ylabel("Energy (J)")
        ax.set_title("Energy vs Context Length"); ax.legend(); ax.grid(alpha=0.3)

        ax = axes[0, 2]
        for cfg in df_perf["config"].unique():
            d = df_perf[df_perf["config"] == cfg].groupby("context_target")["decode_tok_s"].mean()
            ax.plot(d.index, d.values, "D-", label=cfg, lw=2, ms=8)
        ax.set_xlabel("Context Length"); ax.set_ylabel("Throughput (tok/s)")
        ax.set_title("Decode Throughput vs Context Length"); ax.legend(); ax.grid(alpha=0.3)

    if not df_niah.empty:
        ax = axes[1, 0]
        accuracy = df_niah.groupby(["config", "context_target"])["correct"].mean() * 100
        for cfg in df_niah["config"].unique():
            acc = accuracy[cfg]
            ax.plot(acc.index, acc.values, "^-", label=cfg, lw=2, ms=8)
        ax.set_xlabel("Context Length"); ax.set_ylabel("NIAH Accuracy (%)")
        ax.set_title("NIAH Retrieval Accuracy"); ax.legend(); ax.grid(alpha=0.3)
        ax.set_ylim(0, 105)

        ax = axes[1, 1]
        for cfg in df_niah["config"].unique():
            d = df_niah[df_niah["config"] == cfg]
            for ctx in d["context_target"].unique():
                sub = d[d["context_target"] == ctx]
                depth_acc = sub.groupby("needle_depth")["correct"].mean() * 100
                ax.plot(depth_acc.index, depth_acc.values, "o-", label=f"{cfg}@{ctx}")
        ax.set_xlabel("Needle Depth"); ax.set_ylabel("Accuracy (%)")
        ax.set_title("Accuracy vs Needle Depth"); ax.legend(fontsize=8); ax.grid(alpha=0.3)

    if not df_perf.empty and not df_niah.empty:
        ax = axes[1, 2]
        for cfg in df_perf["config"].unique():
            if cfg not in df_niah["config"].values:
                continue
            energy = df_perf[df_perf["config"] == cfg].groupby("context_target")["gross_energy_j"].mean()
            acc = accuracy[cfg] if cfg in accuracy.index else pd.Series()
            if not energy.empty and not acc.empty:
                ax.scatter(energy.values, acc.values, s=100, label=cfg, alpha=0.8, edgecolors="black")
                for ctx, e, a in zip(energy.index, energy.values, acc.values):
                    ax.annotate(str(ctx), (e, a), fontsize=8, ha="center", va="bottom")
        ax.set_xlabel("Energy (J)"); ax.set_ylabel("NIAH Accuracy (%)")
        ax.set_title("Pareto: Energy vs Accuracy"); ax.legend(); ax.grid(alpha=0.3)

    plt.tight_layout()
    fig_path = FIG_DIR / "experiment_results.png"
    plt.savefig(str(fig_path), dpi=150, bbox_inches="tight")
    plt.show()
    print(f"Figure saved to {fig_path}")

plot_results(df_perf, df_niah)

if not df_perf.empty:
    print("\n=== Summary Table (Performance) ===")
    display(df_perf.groupby(["config", "context_target"])[
        ["total_s", "gross_energy_j", "torch_peak_mb", "decode_tok_s", "cache_live_mb"]
    ].mean().round(2))

if not df_niah.empty:
    print("\n=== Summary Table (NIAH) ===")
    summary = df_niah.groupby(["config", "context_target"])["correct"].agg(["mean", "count"])
    summary["mean"] = (summary["mean"] * 100).round(1)
    display(summary.rename(columns={"mean": "accuracy_%", "count": "n"}))